<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_MD2DOCX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

### Converts a structured, machine-readable .md file into human-readable narrative in docx format

In [1]:
!pip install -q python-docx
import json
from pathlib import Path
from IPython.display import Markdown, display
from datetime import datetime
from zoneinfo import ZoneInfo
import pytz
print('ॐ श्री सरस्वत्यै नमः',datetime.now(pytz.timezone('Asia/Calcutta')))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.2 MB/s eta 0:00:00
ॐ श्री सरस्वत्यै नमः 2026-10-08 20:30:42.811099+05:30


In [2]:
!wget -O CentaurLogo.png -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/logo-S.png

!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mosZeitWorld.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosZeitWorld.py
!wget -O mosChanakya.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosChanakya.py
!wget -O mosCentaur.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosCentaur.py
!wget -O mosReporter.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosReporter.py
!wget -O mosDOCx.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosDOCx.py

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *
from mosReporter import *
from mosDOCx import *

In [3]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

# Real-world evidence, LLM roles and prompts
# are stored persistently in Google Drive.

from google.colab import drive
drive.mount('/content/drive')

MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
WF_PATH = f"{MOS_HOME}/WorldFacts"
ZW_PATH = f"{MOS_HOME}/ZeitWorld"
CH_PATH = f"{MOS_HOME}/Chanakya"
CT_PATH = f"{MOS_HOME}/Centaur"
PR_PATH = f"{MOS_HOME}/Reporter"
OP_PATH = f"{MOS_HOME}/Reports"

!ls -l "{OP_PATH}"


OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com
Mounted at /content/drive
total 8
drwx------ 2 root root 4096 Oct  8 07:13 TR1ME4_261008_124321
drwx------ 2 root root 4096 Oct  8 08:00 TR2ME4_261008_133055


In [4]:
!ls -l "{OP_PATH}/TR2ME4_261008_133055"

total 340
-rw------- 1 root root 66420 Oct  8 08:00 MOS_TR2ME4_R01_261008_132630.docx
-rw------- 1 root root 49650 Oct  8 08:00 MOS_TR2ME4_R01_261008_132630.md
-rw------- 1 root root 66416 Oct  8 08:00 MOS_TR2ME4_R02_261008_132755.docx
-rw------- 1 root root 49051 Oct  8 08:00 MOS_TR2ME4_R02_261008_132755.md
-rw------- 1 root root 66377 Oct  8 08:00 MOS_TR2ME4_R03_261008_132922.docx
-rw------- 1 root root 48864 Oct  8 08:00 MOS_TR2ME4_R03_261008_132922.md


In [5]:
def LoadStructuredReport(filename):

    text = Path(filename).read_text(encoding="utf-8")

    json_text = text.split("```json", 1)[1].split("```", 1)[0]
    data = json.loads(json_text)

    W = [WorldState.from_dict(w) for w in data["WorldStates"]]
    E = [Event(**e) for e in data["Events"]]

    return W, E

In [7]:
#StructuredFile = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/Reports/TR2ME4_261008_133055/MOS_TR2ME4_R03_261008_132922.md"
StructuredFile = f"{OP_PATH}/TR2ME4_261008_133055/MOS_TR2ME4_R03_261008_132922.md"

W, E = LoadStructuredReport(StructuredFile)

print("WorldStates:", len(W))
print("Events:", len(E))

for w in W:
    w.show(-1)

for e in E:
    e.show()

WorldStates: 4
Events: 3

World State : W0
-------------------------------------------------------
 0  Oil Price                          100.00
 1  Hormuz Oil Flow                    100.00
 2  Regional Diplomatic Engagement     100.00
 3  Economic Pressure on Iran          100.00
 4  US/Regional Partner Cohesion       100.00
 5  Iran Domestic Stability            100.00
 6  US-Iran Tension                    100.00
 7  Regional Armed Conflict            100.00

World State : W1
-------------------------------------------------------
 0  Oil Price                           94.00
 1  Hormuz Oil Flow                    100.00
 2  Regional Diplomatic Engagement     115.00
 3  Economic Pressure on Iran           98.00
 4  US/Regional Partner Cohesion       100.00
 5  Iran Domestic Stability            103.00
 6  US-Iran Tension                     88.00
 7  Regional Armed Conflict             91.00

World State : W2
-------------------------------------------------------
 0  Oil Price    

In [9]:
# ============================================================
# Load PressReporter Roles and Prompts
#
# Generic Role defines reporting discipline.
# Narrative Prompt produces the human-readable Scenario history.
# Structured Prompt produces the machine-oriented Markdown record.
# ============================================================

PR_Generic_Role = getPrompt(f"{PR_PATH}/PR_Generic_Role_v0.txt")

#PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_v0.txt")
#PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_Brief_v2.txt")
PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_Brief_v3.txt")

PR_Structured   = getPrompt(f"{PR_PATH}/PR_Structured_v0.txt")


# ============================================================
# Instantiate Human Reporter
# ============================================================

Reporter = Reporter_Agent(
    client=client,
    model=model,
    agent_id="Vyasa",
    generic_role=PR_Generic_Role
)

In [10]:
NarrativeReport = Reporter.GenerateReport(W, E, PR_Narrative)

Reporter working ---------- Vyasa 
Model: gpt-5.6-luna | Effort: medium | Input: 8,077 | Output: 838 | Reasoning: 91 | Total: 8,915


In [12]:
_scenarioID = "TR2ME4"
ReportTitle="Middle East Oil Security — Scenario Brief"
now = datetime.now(ZoneInfo("Asia/Kolkata"))
ReportSuffix = now.strftime('%y%m%d_%H%M%S')
NarrativeFile = createNarrativeDocx( NarrativeReport, W, E, _scenarioID,ReportTitle,ReportSuffix )


print(f"Narrative Scenario Report written to: {NarrativeFile}")


Word report created: /content/MOS_TR2ME4_261008_203941.docx
Narrative Scenario Report written to: /content/MOS_TR2ME4_261008_203941.docx
